# Summary Report: Case Study по модулю SQL

**Автор**: Mashrabov Shohzod


In [6]:
import sqlite3
conn = sqlite3.connect(':memory:')  # подключение к базе данных в памяти
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")  # включаем поддержку внешних ключей в SQLite

In [7]:
import pandas as pd

# Чтение данных из Excel в DataFrame
xls = pd.ExcelFile('adventure_works.xlsx')
df_customers = pd.read_excel(xls, sheet_name='Customers')
df_product_category = pd.read_excel(xls, sheet_name='ProductCategory')
df_product_subcategory = pd.read_excel(xls, sheet_name='ProductSubCategory')
df_products = pd.read_excel(xls, sheet_name='Products')
df_territory = pd.read_excel(xls, sheet_name='Territory')
df_sales = pd.read_excel(xls, sheet_name='Sales')

# Приведение названий столбцов в Territory (убираем пробелы)
df_territory.rename(columns={'Territory Key': 'TerritoryKey', 'Group': 'TerritoryGroup'}, inplace=True)

# Конвертация столбцов с датами в текстовый формат YYYY-MM-DD (для корректной загрузки в SQLite)
df_customers['BirthDate'] = pd.to_datetime(df_customers['BirthDate']).dt.strftime('%Y-%m-%d')
df_customers['DateFirstPurchase'] = pd.to_datetime(df_customers['DateFirstPurchase']).dt.strftime('%Y-%m-%d')
df_products['StartDate'] = pd.to_datetime(df_products['StartDate']).dt.strftime('%Y-%m-%d')
df_products['EndDate'] = pd.to_datetime(df_products['EndDate']).dt.strftime('%Y-%m-%d')
df_sales['OrderDate'] = pd.to_datetime(df_sales['OrderDate']).dt.strftime('%Y-%m-%d')

# Заменяем NaN на None для корректной вставки NULL в базу
dfs = [df_customers, df_product_category, df_product_subcategory, df_products, df_territory, df_sales]
for i, df in enumerate(dfs):
    dfs[i] = df.where(pd.notnull(df), None)
df_customers, df_product_category, df_product_subcategory, df_products, df_territory, df_sales = dfs

# Определение SQL-скриптов для создания таблиц
create_customers = """
CREATE TABLE Customers (
    CustomerKey INTEGER PRIMARY KEY,
    GeographyKey INTEGER,
    Name TEXT,
    BirthDate TEXT,
    MaritalStatus TEXT,
    Gender TEXT,
    YearlyIncome INTEGER,
    NumberChildrenAtHome INTEGER,
    Occupation TEXT,
    HouseOwnerFlag INTEGER,
    NumberCarsOwned INTEGER,
    AddressLine1 TEXT,
    AddressLine2 TEXT,
    Phone TEXT,
    DateFirstPurchase TEXT
);
"""
create_product_category = """
CREATE TABLE ProductCategory (
    ProductCategoryKey INTEGER PRIMARY KEY,
    ProductCategoryAlternateKey INTEGER,
    EnglishProductCategoryName TEXT,
    SpanishProductCategoryName TEXT,
    FrenchProductCategoryName TEXT
);
"""
create_product_subcategory = """
CREATE TABLE ProductSubCategory (
    ProductSubcategoryKey INTEGER PRIMARY KEY,
    ProductSubcategoryAlternateKey INTEGER,
    EnglishProductSubcategoryName TEXT,
    SpanishProductSubcategoryName TEXT,
    FrenchProductSubcategoryName TEXT,
    ProductCategoryKey INTEGER,
    FOREIGN KEY (ProductCategoryKey) REFERENCES ProductCategory(ProductCategoryKey)
);
"""
create_products = """
CREATE TABLE Products (
    ProductKey INTEGER PRIMARY KEY,
    ProductSubcategoryKey INTEGER,
    ProductName TEXT,
    StandardCost REAL,
    Color TEXT,
    SafetyStockLevel INTEGER,
    ListPrice REAL,
    Size TEXT,
    SizeRange TEXT,
    Weight REAL,
    DaysToManufacture INTEGER,
    ProductLine TEXT,
    DealerPrice REAL,
    Class TEXT,
    ModelName TEXT,
    Description TEXT,
    StartDate TEXT,
    EndDate TEXT,
    Status TEXT,
    FOREIGN KEY (ProductSubcategoryKey) REFERENCES ProductSubCategory(ProductSubcategoryKey)
);
"""
create_territory = """
CREATE TABLE Territory (
    TerritoryKey INTEGER PRIMARY KEY,
    Region TEXT,
    Country TEXT,
    TerritoryGroup TEXT
);
"""
create_sales = """
CREATE TABLE Sales (
    ProductKey INTEGER,
    OrderDate TEXT,
    OrderDateKey INTEGER,
    CustomerKey INTEGER,
    SalesTerritoryKey INTEGER,
    SalesOrderNumber TEXT,
    SalesOrderLineNumber INTEGER,
    OrderQuantity INTEGER,
    UnitPrice REAL,
    ExtendedAmount REAL,
    UnitPriceDiscountPct REAL,
    DiscountAmount REAL,
    ProductStandardCost REAL,
    TotalProductCost REAL,
    SalesAmount REAL,
    TaxAmt REAL,
    Freight REAL,
    RegionMonthID INTEGER,
    PRIMARY KEY (SalesOrderNumber, SalesOrderLineNumber),
    FOREIGN KEY (ProductKey) REFERENCES Products(ProductKey),
    FOREIGN KEY (CustomerKey) REFERENCES Customers(CustomerKey),
    FOREIGN KEY (SalesTerritoryKey) REFERENCES Territory(TerritoryKey)
);
"""
# Выполнение создания таблиц
cur.execute(create_customers)
cur.execute(create_product_category)
cur.execute(create_product_subcategory)
cur.execute(create_products)
cur.execute(create_territory)
cur.execute(create_sales)
conn.commit()

# Подготовка SQL-запросов для вставки данных
insert_customers = """INSERT INTO Customers VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);"""
insert_product_category = """INSERT INTO ProductCategory VALUES (?, ?, ?, ?, ?);"""
insert_product_subcategory = """INSERT INTO ProductSubCategory VALUES (?, ?, ?, ?, ?, ?);"""
insert_products = """INSERT INTO Products VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);"""
insert_territory = """INSERT INTO Territory VALUES (?, ?, ?, ?);"""
insert_sales = """INSERT INTO Sales VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);"""

# Вставка данных из DataFrame в таблицы через executemany
cur.executemany(insert_customers, [tuple(row) for row in df_customers.itertuples(index=False, name=None)])
cur.executemany(insert_product_category, [tuple(row) for row in df_product_category.itertuples(index=False, name=None)])
cur.executemany(insert_product_subcategory, [tuple(row) for row in df_product_subcategory.itertuples(index=False, name=None)])
cur.executemany(insert_products, [tuple(row) for row in df_products.itertuples(index=False, name=None)])
cur.executemany(insert_territory, [tuple(row) for row in df_territory.itertuples(index=False, name=None)])
cur.executemany(insert_sales, [tuple(row) for row in df_sales.itertuples(index=False, name=None)])
conn.commit()

In [8]:
tables = ["Customers", "ProductCategory", "ProductSubCategory", "Products", "Territory", "Sales"]
for table in tables:
    count = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

Customers: 18484 rows
ProductCategory: 4 rows
ProductSubCategory: 37 rows
Products: 397 rows
Territory: 11 rows
Sales: 60398 rows


In [9]:
query = """
SELECT 
    Occupation AS occupation,
    COUNT(*) AS number_of_customers,
    ROUND(AVG(YearlyIncome), 0) AS avg_income
FROM Customers
GROUP BY Occupation
ORDER BY number_of_customers DESC;
"""
df_income_segmentation = pd.read_sql_query(query, conn)
df_income_segmentation

,occupation,number_of_customers,avg_income
0,Professional,5520,74185.0
1,Skilled Manual,4577,51715.0
2,Management,3075,92325.0
3,Clerical,2928,30710.0
4,Manual,2384,16451.0


In [10]:
query = """
SELECT 
    CASE WHEN NumberChildrenAtHome > 0 THEN 'Yes' ELSE 'No' END AS has_children,
    ROUND(COUNT(*) * 1.0 / (SELECT COUNT(*) FROM Customers) * 100, 2) AS pct_of_customer_base
FROM Customers
GROUP BY CASE WHEN NumberChildrenAtHome > 0 THEN 'Yes' ELSE 'No' END;
"""
df_family = pd.read_sql_query(query, conn)
df_family

,has_children,pct_of_customer_base
0,No,60.14
1,Yes,39.86


In [11]:
query = """
SELECT 
    s.CustomerKey AS customer_key,
    c.Name AS customer_name,
    ROUND(SUM(s.SalesAmount), 2) AS total_purchase
FROM Sales s
JOIN Customers c ON s.CustomerKey = c.CustomerKey
GROUP BY s.CustomerKey, c.Name
ORDER BY SUM(s.SalesAmount) DESC
LIMIT 10;
"""
df_top_customers = pd.read_sql_query(query, conn)
df_top_customers

,customer_key,customer_name,total_purchase
0,12301,Nichole Nara,13295.38
1,12132,Kaitlyn Henderson,13294.27
2,12308,Margaret He,13269.27
3,12131,Randall Dominguez,13265.99
4,12300,Adriana Gonzalez,13242.70
5,12321,Rosa Hu,13215.65
6,12124,Brandi Gill,13195.64
7,12307,Brad She,13173.19
8,12296,Francisco Sara,13164.64
9,11433,Maurice Shan,12909.67


In [12]:
query = """
SELECT 
    CAST(strftime('%Y', s.OrderDate) AS INTEGER) AS year,
    c.MaritalStatus AS marital_status,
    ROUND(AVG(s.SalesAmount), 2) AS avg_sales_amount
FROM Sales s
JOIN Customers c ON s.CustomerKey = c.CustomerKey
GROUP BY year, c.MaritalStatus
ORDER BY year, c.MaritalStatus;
"""
df_marital = pd.read_sql_query(query, conn)
df_marital

,year,marital_status,avg_sales_amount
0,2001,M,3245.03
1,2001,S,3203.84
2,2002,M,2397.07
3,2002,S,2482.13
4,2003,M,378.56
5,2003,S,427.78
6,2004,M,290.64
7,2004,S,318.05


In [13]:
query = """
SELECT 
    CAST(year AS INTEGER) AS year,
    CAST(month AS INTEGER) AS monthkey,
    CASE month 
        WHEN '01' THEN 'January' WHEN '02' THEN 'February'
        WHEN '03' THEN 'March'   WHEN '04' THEN 'April'
        WHEN '05' THEN 'May'     WHEN '06' THEN 'June'
        WHEN '07' THEN 'July'    WHEN '08' THEN 'August'
        WHEN '09' THEN 'September' WHEN '10' THEN 'October'
        WHEN '11' THEN 'November'  WHEN '12' THEN 'December'
    END AS month_name,
    sales_count,
    sales_amount
FROM (
    SELECT 
        strftime('%Y', OrderDate) AS year,
        strftime('%m', OrderDate) AS month,
        COUNT(*) AS sales_count,
        ROUND(SUM(SalesAmount), 2) AS sales_amount
    FROM Sales
    WHERE strftime('%Y', OrderDate) IN ('2003','2004')
    GROUP BY strftime('%Y', OrderDate), strftime('%m', OrderDate)
)
ORDER BY CAST(year AS INTEGER), CAST(month AS INTEGER);
"""
df_monthly_sales = pd.read_sql_query(query, conn)
df_monthly_sales.head(12)  # вывод первых 12 строк (месяцы 2003 года)

,year,monthkey,month_name,sales_count,sales_amount
0,2003,1,January,244,438865.17
1,2003,2,February,272,489090.34
2,2003,3,March,272,485574.79
3,2003,4,April,294,506399.27
4,2003,5,May,335,562772.56
5,2003,6,June,321,554799.23
6,2003,7,July,1411,886668.84
7,2003,8,August,3819,847413.51
8,2003,9,September,3885,1010258.13
9,2003,10,October,4146,1080449.58


In [ ]:
query = """
SELECT 
    t.Region AS region,
    COUNT(*) AS sales_count,
    ROUND(SUM(s.SalesAmount), 2) AS sales_amount
FROM Sales s
JOIN Territory t ON s.SalesTerritoryKey = t.TerritoryKey
GROUP BY t.Region
ORDER BY sales_amount DESC;
"""
df_region_sales = pd.read_sql_query(query, conn)
df_region_sales

,region,sales_count,sales_amount
0,Australia,13345,9061000.58
1,Southwest,12265,5718150.81
2,Northwest,8993,3649866.55
3,United Kingdom,6906,3391712.21
4,Germany,5625,2894312.34
5,France,5558,2644017.71
6,Canada,7620,1977844.86
7,Southeast,39,12238.85
8,Northeast,27,6532.47
9,Central,20,3000.83


In [ ]:
query = """
SELECT 
    CAST(strftime('%Y', s.OrderDate) AS INTEGER) AS year,
    p.ProductKey AS product_key,
    sub.ProductCategoryKey AS product_category_key,
    pc.EnglishProductCategoryName AS english_product_category_name,
    ROUND(SUM(s.SalesAmount), 2) AS sales_amount,
    ROUND(100.0 * SUM(s.SalesAmount) / 
          (SELECT SUM(SalesAmount) 
           FROM Sales s2 
           WHERE CAST(strftime('%Y', s2.OrderDate) AS INTEGER) = CAST(strftime('%Y', s.OrderDate) AS INTEGER)
          ), 2) AS pct_of_total_sales
FROM Sales s
JOIN Products p ON s.ProductKey = p.ProductKey
JOIN ProductSubCategory sub ON p.ProductSubcategoryKey = sub.ProductSubcategoryKey
JOIN ProductCategory pc ON sub.ProductCategoryKey = pc.ProductCategoryKey
GROUP BY year, p.ProductKey
ORDER BY year, pct_of_total_sales DESC;
"""
df_product_share = pd.read_sql_query(query, conn)
df_product_share.head(10)  # первые 10 строк результата

,year,product_key,product_category_key,english_product_category_name,sales_amount,pct_of_total_sales
0,2001,310,1,Bikes,593992.82,18.19
1,2001,312,1,Bikes,547475.31,16.76
2,2001,311,1,Bikes,500957.80,15.34
3,2001,314,1,Bikes,486644.72,14.90
4,2001,313,1,Bikes,472331.64,14.46
5,2001,344,1,Bikes,98599.71,3.02
6,2001,350,1,Bikes,97874.71,3.00
7,2001,346,1,Bikes,84999.75,2.60
8,2001,351,1,Bikes,70874.79,2.17
9,2001,348,1,Bikes,70874.79,2.17


In [ ]:
query = """
SELECT 
    p.ProductKey AS product_key,
    p.ProductName AS product_name,
    pc.EnglishProductCategoryName AS english_product_category_name,
    ROUND(SUM(s.SalesAmount), 2) AS sales_amount
FROM Sales s
JOIN Products p ON s.ProductKey = p.ProductKey
JOIN ProductSubCategory sub ON p.ProductSubcategoryKey = sub.ProductSubcategoryKey
JOIN ProductCategory pc ON sub.ProductCategoryKey = pc.ProductCategoryKey
GROUP BY p.ProductKey
ORDER BY SUM(s.SalesAmount) DESC
LIMIT 5;
"""
df_top_products = pd.read_sql_query(query, conn)
df_top_products

,product_key,product_name,english_product_category_name,sales_amount
0,312,"Road-150 Red, 48",Bikes,1205876.99
1,310,"Road-150 Red, 62",Bikes,1202298.72
2,313,"Road-150 Red, 52",Bikes,1080637.54
3,314,"Road-150 Red, 56",Bikes,1055589.65
4,311,"Road-150 Red, 44",Bikes,1005493.87


In [ ]:
query = """
SELECT 
    CAST(year AS INTEGER) AS year,
    CAST(month AS INTEGER) AS monthkey,
    CASE month 
         WHEN '01' THEN 'January' WHEN '02' THEN 'February'
         WHEN '03' THEN 'March'   WHEN '04' THEN 'April'
         WHEN '05' THEN 'May'     WHEN '06' THEN 'June'
         WHEN '07' THEN 'July'    WHEN '08' THEN 'August'
         WHEN '09' THEN 'September' WHEN '10' THEN 'October'
         WHEN '11' THEN 'November'  WHEN '12' THEN 'December'
    END AS month_name,
    product_key,
    product_name,
    ROUND(sales_amount, 2) AS sales_amount,
    ROUND(total_product_cost, 2) AS total_product_cost,
    ROUND(tax_amt, 2) AS tax_amt,
    ROUND(freight, 2) AS freight,
    ROUND(sales_amount - total_product_cost - tax_amt - freight, 2) AS margin,
    ROUND(100.0 * (sales_amount - total_product_cost - tax_amt - freight) / sales_amount, 2) AS margin_pct
FROM (
    SELECT 
        strftime('%Y', OrderDate) AS year,
        strftime('%m', OrderDate) AS month,
        p.ProductKey AS product_key,
        p.ProductName AS product_name,
        SUM(s.SalesAmount) AS sales_amount,
        SUM(s.TotalProductCost) AS total_product_cost,
        SUM(s.TaxAmt) AS tax_amt,
        SUM(s.Freight) AS freight
    FROM Sales s
    JOIN Products p ON s.ProductKey = p.ProductKey
    GROUP BY strftime('%Y', OrderDate), strftime('%m', OrderDate), p.ProductKey
)
ORDER BY CAST(year AS INTEGER), CAST(month AS INTEGER), product_key;
"""
df_margin = pd.read_sql_query(query, conn)
df_margin.head(10)  # первые 10 строк результата

,year,monthkey,month_name,product_key,product_name,sales_amount,total_product_cost,tax_amt,freight,margin,margin_pct
0,2001,7,July,310,"Road-150 Red, 62",78721.94,47768.47,6297.76,1968.05,22687.66,28.82
1,2001,7,July,311,"Road-150 Red, 44",82300.21,49939.77,6584.02,2057.51,23718.92,28.82
2,2001,7,July,312,"Road-150 Red, 48",100191.56,60796.24,8015.32,2504.79,28875.21,28.82
3,2001,7,July,313,"Road-150 Red, 52",42939.24,26055.53,3435.14,1073.48,12375.09,28.82
4,2001,7,July,314,"Road-150 Red, 56",53674.05,32569.41,4293.92,1341.85,15468.86,28.82
5,2001,7,July,324,"Road-650 Red, 62",699.10,413.15,55.93,17.48,212.55,30.40
6,2001,7,July,326,"Road-650 Red, 44",1398.20,826.29,111.86,34.95,425.09,30.40
7,2001,7,July,328,"Road-650 Red, 48",699.10,413.15,55.93,17.48,212.55,30.40
8,2001,7,July,330,"Road-650 Red, 52",1398.20,826.29,111.86,34.95,425.09,30.40
9,2001,7,July,332,"Road-650 Black, 58",2097.29,1239.44,167.78,52.43,637.64,30.40


In [ ]:
query = """
WITH TopCategories AS (
    SELECT sub.ProductCategoryKey AS category_key
    FROM Sales s
    JOIN Products p ON s.ProductKey = p.ProductKey
    JOIN ProductSubCategory sub ON p.ProductSubcategoryKey = sub.ProductSubcategoryKey
    GROUP BY sub.ProductCategoryKey
    ORDER BY SUM(s.SalesAmount) DESC
    LIMIT 2
)
SELECT 
    curr.year,
    curr.quarter_id,
    curr.category_key AS product_category_key,
    pc.EnglishProductCategoryName AS english_product_category_name,
    curr.quarter_sales_amount,
    ROUND(
        100.0 * (curr.quarter_sales_amount - prev.prev_quarter_sales_amount) / prev.prev_quarter_sales_amount,
        2
    ) AS quarter_over_quarter_growth_pct
FROM (
    SELECT 
        CAST(strftime('%Y', OrderDate) AS INTEGER) AS year,
        CASE 
            WHEN CAST(strftime('%m', OrderDate) AS INTEGER) BETWEEN 1 AND 3 THEN 1
            WHEN CAST(strftime('%m', OrderDate) AS INTEGER) BETWEEN 4 AND 6 THEN 2
            WHEN CAST(strftime('%m', OrderDate) AS INTEGER) BETWEEN 7 AND 9 THEN 3
            WHEN CAST(strftime('%m', OrderDate) AS INTEGER) BETWEEN 10 AND 12 THEN 4
        END AS quarter_id,
        sub.ProductCategoryKey AS category_key,
        SUM(s.SalesAmount) AS quarter_sales_amount
    FROM Sales s
    JOIN Products p ON s.ProductKey = p.ProductKey
    JOIN ProductSubCategory sub ON p.ProductSubcategoryKey = sub.ProductSubcategoryKey
    WHERE sub.ProductCategoryKey IN (SELECT category_key FROM TopCategories)
    GROUP BY CAST(strftime('%Y', OrderDate) AS INTEGER), quarter_id, sub.ProductCategoryKey
) AS curr
LEFT JOIN (
    SELECT 
        CAST(strftime('%Y', OrderDate) AS INTEGER) AS year,
        CASE 
            WHEN CAST(strftime('%m', OrderDate) AS INTEGER) BETWEEN 1 AND 3 THEN 1
            WHEN CAST(strftime('%m', OrderDate) AS INTEGER) BETWEEN 4 AND 6 THEN 2
            WHEN CAST(strftime('%m', OrderDate) AS INTEGER) BETWEEN 7 AND 9 THEN 3
            WHEN CAST(strftime('%m', OrderDate) AS INTEGER) BETWEEN 10 AND 12 THEN 4
        END AS quarter_id,
        sub.ProductCategoryKey AS category_key,
        SUM(s.SalesAmount) AS prev_quarter_sales_amount
    FROM Sales s
    JOIN Products p ON s.ProductKey = p.ProductKey
    JOIN ProductSubCategory sub ON p.ProductSubcategoryKey = sub.ProductSubcategoryKey
    WHERE sub.ProductCategoryKey IN (SELECT category_key FROM TopCategories)
    GROUP BY CAST(strftime('%Y', OrderDate) AS INTEGER), quarter_id, sub.ProductCategoryKey
) AS prev
ON curr.category_key = prev.category_key
   AND (
       (curr.year = prev.year AND curr.quarter_id = prev.quarter_id + 1)
       OR (curr.year = prev.year + 1 AND curr.quarter_id = 1 AND prev.quarter_id = 4)
   )
JOIN ProductCategory pc ON curr.category_key = pc.ProductCategoryKey
WHERE curr.category_key IN (SELECT category_key FROM TopCategories)
ORDER BY curr.category_key, curr.year, curr.quarter_id;
"""

df = pd.read_sql_query(query, conn)
df.head()

,year,quarter_id,product_category_key,english_product_category_name,quarter_sales_amount,quarter_over_quarter_growth_pct
0,2001,3,1,Bikes,1.453523e+06,NaN
1,2001,4,1,Bikes,1.812851e+06,24.72
2,2002,1,1,Bikes,1.791698e+06,-1.17
3,2002,2,1,Bikes,2.014012e+06,12.41
4,2002,3,1,Bikes,1.396834e+06,-30.64


In [ ]:
query = """
SELECT 
    CAST(strftime('%Y', OrderDate) AS INTEGER) AS year,
    CASE strftime('%w', OrderDate)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS day_name,
    CASE strftime('%w', OrderDate)
        WHEN '0' THEN 'Yes' 
        WHEN '6' THEN 'Yes'
        ELSE 'No'
    END AS is_weekend,
    ROUND(SUM(SalesAmount), 2) AS sales_amount
FROM Sales
GROUP BY year, day_name
ORDER BY year, CASE strftime('%w', OrderDate)
                 WHEN '0' THEN 7
                 ELSE CAST(strftime('%w', OrderDate) AS INTEGER)
               END;
"""
df_dayofweek = pd.read_sql_query(query, conn)
df_dayofweek

,year,day_name,is_weekend,sales_amount
0,2001,Monday,No,447197.10
1,2001,Tuesday,No,433609.42
2,2001,Wednesday,No,435300.15
3,2001,Thursday,No,450281.09
4,2001,Friday,No,468724.50
5,2001,Saturday,Yes,505234.58
6,2001,Sunday,Yes,526026.82
7,2002,Monday,No,919784.70
8,2002,Tuesday,No,941228.27
9,2002,Wednesday,No,1004132.60
